# Music Recommender API worker

Create a **Worker API** key in the Music Recommender UI, set or enter `WORKER_URL` below, then run the cell. Python 3.11 is recommended; on Python 3.12+ the cell applies a packaging-only compatibility patch for the pinned OpenL3/resampy source releases without changing model code or versions. Worker JSON events are streamed line-by-line into the cell output. The cell performs a one-track dry run by default; exit code 2 is expected when unembedded tracks remain, but the final `run_summary` must be checked for failures. Remove `--dry-run` and `--limit 1` only when you are ready to write embeddings. The key is prompted for rather than stored in notebook source.

Tune throughput with `infer_batch_size` below (or set `EMBEDDING_INFER_BATCH_SIZE`). It sets the OpenL3 predict batch: 64 is safe on CPU, and 128 or 256 usually helps on a GPU. Do not raise `--batch-size`; that is the keyset page size and the API caps it at 32. Exit code 64 means the arguments or environment were rejected and nothing was processed.

In [ ]:
import getpass
import os
import subprocess
import sys

if sys.version_info < (3, 11):
    raise RuntimeError("The worker requires Python 3.11 or newer.")

repo_url = "https://github.com/poesterlin/music-recommender.git"
repo_dir = "music-recommender"
if not os.path.isdir(os.path.join(repo_dir, "embeddings")):
    subprocess.run(["git", "clone", "--depth", "1", repo_url, repo_dir], check=True)
else:
    subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
os.chdir(repo_dir)
if sys.version_info >= (3, 12):
    subprocess.run([sys.executable, "embeddings/install_python312.py"], check=True)
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "embeddings/requirements.txt"], check=True)

os.environ["WORKER_URL"] = os.environ.get("WORKER_URL") or input("Worker URL (for example, https://recommender.example.com): ").strip()
os.environ["WORKER_TOKEN"] = getpass.getpass("Paste worker API key: ")
# Only inference batching is tuned here. Raise it on a GPU; 64 is safe on CPU.
infer_batch_size = os.environ.get("EMBEDDING_INFER_BATCH_SIZE", "64")
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
process = subprocess.Popen(
    [
        sys.executable,
        "embeddings/worker.py",
        "--source-mode",
        "api",
        "--duration",
        "60",
        "--infer-batch-size",
        infer_batch_size,
        "--dry-run",
        "--limit",
        "1",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="", flush=True)
returncode = process.wait()
if returncode == 0:
    print("Worker finished with no pending tracks.")
elif returncode == 2:
    print("Worker stopped with exit code 2; this is expected when pending tracks remain. Check run_summary for failures.")
elif returncode == 64:
    raise RuntimeError("Worker configuration is invalid (exit 64). Fix the arguments or environment above; nothing was processed.")
else:
    raise subprocess.CalledProcessError(returncode, process.args)